# FDS Challenge: My Notebook

## 0. Configuration

In [ ]:
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

from scipy.optimize import minimize
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


DEBUG = False

pd.set_option("display.max_columns", None)  # keep it! it's usefull to make pandas print also the null columns
pd.set_option("display.width", None)


# --- PATHS ---
TRAIN_FILE_PATH        = "../dataset/train.jsonl"
TEST_FILE_PATH         = "../dataset/test.jsonl"
TYPE_CHART_PATH        = "../dataset/type_chart_gen2.csv"
SAMPLE_SUBMISSION_PATH = "../dataset/sample_submission.csv"
SUBMISSION_PATH        = "submission.csv"
REPORT_DIR             = "reports"  # One timestamped Markdown report per run.

Path("../temp").mkdir(exist_ok=True)


# --- FEATURES ---
TEAM_SIZE = 6

STATISTICS = ["hp", "atk", "def", "spa", "spd", "spe"]
HP_PCT_LOST_INTERVALS = ["0_10", "10_20", "20_30"]
MULTIPLIERS = [0, 0.25, 0.5, 1, 2, 4]

STATUSES_WEIGHTS = {
    "fnt": 6,
    "slp": 2,
    "frz": 4,
    "tox": 3,
    "psn": 1,
    "par": 2,
    "brn": 2
}

BOOSTS_WEIGHTS = {
    "atk": 2,
    "def": 1,
    "spa": 2,
    "spd": 1,
    "spe": 2
}

MULTS_WEIGHTS = {
    0:      -4,
    0.25: -1.5,
    0.5:  -0.5,
    1:       0,
    2:     1.5,
    4:       3
}

EFFECT_WEIGHTS = {
    "attract":      -0.5,
    "beatup":        0.3,
    "bide":          0.4,
    "bind":         -0.4,
    "clamp":        -1.0,
    "confusion":    -0.8,
    "curse":        -0.8,
    "destinybond":   0.6,
    "disable":      -0.4,
    "encore":       -1.2,
    "endure":       -0.5,
    "firespin":     -0.4,
    "focusband":    -0.8,
    "focusenergy":  -0.4,
    "foresight":    -0.2,
    "futuresight":  -0.7,
    "leechseed":    -0.8,
    "lockon":        0.2,
    "magnitude":     0.3,
    "mimic":        -0.6,
    "mindreader":   -0.5,
    "mysteryberry": -0.3,
    "nightmare":    -1.0,
    "noeffect":      0.0,
    "perish":        0.0,
    "protect":       0.0,
    "pursuit":      -1.5,
    "quickclaw":    -0.6,
    "safeguard":     0.6,
    "spite":        -0.3,
    "substitute":    1.0,
    "trapped":      -0.9,
    "typechange":    0.3,
    "whirlpool":    -0.4,
    "wrap":         -0.4
}

# --- HEADCOUNT COMPARISON FEATURE LISTS ---

ALLOWED_FEATURES = {
    # 1. Reconstructed Headcount & Bulk
    "p1_fnt",
    "p2_confirmed_kos",
    "p2_estimated_true_kos",
    "diff_estimated_pokemon_alive",
    "p2_known_remaining_hp_sum",
    "diff_lst_hp_mean",
    "p1_lst_hp_mean",
    "p2_lst_hp_mean",
    # 2. HP Intervals & Loss (Includes Turns 0-10 Early Tempo)
    "diff_hp_pct_lost_0_10",
    "p1_hp_pct_lost_0_10",
    "p2_hp_pct_lost_0_10",
    "diff_hp_pct_lost_10_20",
    "diff_hp_pct_lost_20_30",
    "p1_hp_pct_lost_20_30",
    "p2_hp_pct_lost_20_30",
    "diff_tot_hp_lost",
    # 3. Statuses (Weighted Summaries)
    "diff_lst_statuses_wmean",
    "p1_lst_statuses_wmean",
    "p2_lst_statuses_wmean",
    "diff_statuses_wmean",
    # 4. Stat Stages & Boosts (Weighted Summaries)
    "diff_lst_boosts_wmean",
    "p1_lst_boosts_wmean",
    "p2_lst_boosts_wmean",
    "diff_boosts_wmean",
    "p1_boosts_wmean",
    "p2_boosts_wmean",
    # 5. Type Effectiveness & Real Attack Multipliers
    "p1_mult_wmean",
    "p2_mult_wmean",
    "diff_0_mult",
    "diff_2_mult",
    "diff_mult_wmean",
    "diff_mult_per_atk",
    "diff_eff_net_mean",
    "diff_lst_effects_wmean",
    "p1_lst_effects_wmean",
    # 6. Core Defensive Base Stats & Differentials
    "diff_mean_def",
    "diff_mean_hp",
    "p1_mean_def",
    "p1_mean_spa",
    "p1_mean_spd",
    "p1_mean_spe",
    "p1_min_base_hp",
    "p1_min_base_atk",
    "p1_min_base_def",
    "p1_min_base_spa",
    "p1_min_base_spd",
    "p1_min_base_spe",
    "p1_max_base_hp",
    "p1_max_base_atk",
    "p1_max_base_def",
    "p1_max_base_spa",
    "p1_max_base_spd",
    "p1_max_base_spe",
    "p1_std_base_hp",
    "p1_std_base_atk",
    "p1_std_base_def",
    "p1_std_base_spa",
    "p1_std_base_spd",
    "p1_std_base_spe",
    "p2_sum_hp",
    "p2_sum_atk",
    "p2_sum_def",
    "p2_sum_spa",
    "p2_sum_spd",
    "p2_sum_spe",
    "p2_mean_hp",
    "diff_mean_atk",
    "diff_mean_spa",
    "diff_mean_spd",
    "diff_mean_spe",
    "diff_mean_bst",
    # 7. Held Items & Detections
    "p1_item_leftovers_count",
    "p1_item_has_all_leftovers",
    "p1_item_unique_count",
    "p1_item_junk_count",
    "p1_item_status_berry_count",
    "p1_item_type_boost_count",
    "p1_item_rng_count",
    "p1_has_thickclub_marowak",
    "p2_detected_leftovers_count",
    "diff_leftovers_count",
    "diff_junk_items_count",
    # 8. Deployed Counts, Active Level & Effective Bulk
    "p1_active_wmean_lvl",
    "p1_deployed_count",
    "p2_deployed_count",
    "diff_deployed_count",
    "p1_rem_effective_hp_ratio",
    "p2_rem_effective_hp_ratio",
    "diff_rem_effective_hp_ratio",
    # 9. Tactical Tempo, Hazards & RestTalk
    "diff_moves_used",
    "diff_switches",
    "p1_eff_power_mean",
    "p2_eff_power_mean",
    "diff_eff_power_mean",
    "diff_rest_used",
    "diff_sleeptalk_used",
    "diff_spikes_active",
    "diff_phaze_used",
    # 10. Macro Team Typing & Defensive Profiling
    "diff_type_diversity",
    "diff_type_weakness_exposure",
    "diff_shared_weakness_count",
    "p2_stat_coverage",
    "p2_detected_rng_count",
}

# --- HELD ITEMS DICTIONARIES & METRIC PRIORS ---
JUNK_ITEMS = {
    "berryjuice", "berry", "noitem", "pokeball", "firestone",
    "energypowder", "aspearberry", "gengarite", "alakazite", "blackbelt"
}

STATUS_BERRIES = {
    "miracleberry", "mintberry", "przcureberry", "psncureberry",
    "bitterberry", "lumberry", "chestoberry"
}

TYPE_BOOST_ITEMS = {
    "charcoal", "magnet", "blackbelt", "pinkbow", "polkadotbow",
    "mysticwater", "hardstone", "poisonbarb", "softsand", "silverpowder",
    "blackglasses", "sharpbeak", "spelltag", "miracleseed", "twistedspoon",
    "nevermeltice", "dragonscale", "dragonfang", "metalcoat"
}

RNG_ITEMS = {
    "berserkgene", "scopelens", "quickclaw", "focusband", "brightpowder", "kingsrock"
}

HEALING_MOVES = {
    "rest", "recover", "softboiled", "milkdrink", "synthesis",
    "morningsun", "moonlight", "drainpunch", "megadrain", "gigadrain", "leechlife"
}

STAB = 1.5

# --- FEATURE SELECTION ---
# Selection searches ALL_FEATURES; every listed column must be extracted.
# It takes priority over the manual ALLOWED_FEATURES filter.
USE_FEATURE_SELECTION = True
USE_ALLOWED_FEATURES = False
ALLOWED_FEATURES = []  # Paste a selected subset here for later manual runs.

"lightgbm":
A gradient boosting model that builds trees one after another, each correcting the errors of the previous ones.
It is usually the most accurate on tabular data.

# Run 2: deep, expensive refinement; first reduce ALL_FEATURES to ~20-40
# candidates from run 1 (winner plus promising alternatives). Extraction stays unchanged.
# USE_ALLOWED_FEATURES does not restrict this search; edit ALL_FEATURES itself.
# Suggested overrides; keep the other settings above:
# SELECTION_DROP_FRACTION = 0.1
# SELECTION_MAX_ITERATIONS = 2000
# SELECTION_SHAP_SAMPLE_SIZE = 2048
# SELECTION_ACCURACY_TOLERANCE = 0.001  # Within 0.1 percentage points of the best inner score.

"catboost":
Gradient boosting with symmetric (oblivious) trees, useful as a complementary tabular model.

"ensemble":
A soft-voting combination of LightGBM, XGBoost, CatBoost, and Logistic Regression.
Each branch standardizes features within the training fold before fitting.
"""
MODEL_TYPE = "logistic_regression"
RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 3
EARLY_STOPPING_ROUNDS = 100
EARLY_STOPPING_FRACTION = 0.15
DECISION_THRESHOLD = 0.5

# String/category columns are detected automatically. Add integer-coded
# categorical columns here when extending the feature lists.
CATEGORICAL_FEATURES = []

# Starting settings, not tuned results. Early stopping selects the tree count
# using Logloss; compare feature experiments using validation accuracy.
CATBOOST_PARAMS = {
    "iterations": 2000,
    "depth": 6,
    "learning_rate": 0.03,
    "l2_leaf_reg": 5.0,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "task_type": "CPU",
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": False,
}

## 1. Loading Data

In [ ]:
# --- LOAD DATA ---
def load_data(file_path: str) -> list:
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data  = load_data(TEST_FILE_PATH)

### Pokemon Stats & Moves Type

In [ ]:
# --- POKEMON TYPE MAP ---
def create_stats_map(train_data: list, test_data: list) -> dict:
    """
    Build a species stats/type lookup from supplied battle details, without labels
    """

    print("Creating Pokémon stats lookup...")
    stats_map = {}
    for data in (train_data, test_data):
        for battle in data:

            # Saving pokemon stats for p1 team
            for pokemon in battle["p1_team_details"]:
                if pokemon["name"] not in stats_map:
                    stats_map[pokemon["name"]] = {
                        "base_stats": {
                            "base_hp": pokemon["base_hp"],
                            "base_atk": pokemon["base_atk"],
                            "base_def": pokemon["base_def"],
                            "base_spa": pokemon["base_spa"],
                            "base_spd": pokemon["base_spd"],
                            "base_spe": pokemon["base_spe"],
                        },
                        "types": pokemon["types"]
                    }
                
            # Retreive p2 lead stats
            if battle["p2_lead_details"]["name"] not in stats_map:
                lead = battle["p2_lead_details"]
                stats_map[lead["name"]] = {
                    "base_stats": {
                        "base_hp": lead["base_hp"],
                        "base_atk": lead["base_atk"],
                        "base_def": lead["base_def"],
                        "base_spa": lead["base_spa"],
                        "base_spd": lead["base_spd"],
                        "base_spe": lead["base_spe"],
                    },
                    "types": lead["types"]
                }

    print(f"SUCCESS: found {len(stats_map)} pokemon.")
    return stats_map


# --- MOVES TYPE JSON ---
def normalize_type_chart(type_chart: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy with trimmed column names and lowercase, trimmed types; reject missing types.
    """

    # Clear columns headers
    type_chart.columns = type_chart.columns.astype(str).str.strip()

    # Clear columns values
    for col in ["attacking_type", "defending_type"]:
        # If a value is Nan break the run
        if type_chart[col].isna().any():
            nan_rows = type_chart[type_chart[col].isna()].index.tolist()
            raise ValueError(f"ERROR: found NaN value in column '{col}' at rows {nan_rows}.")

        # Remove white spaces and lower
        type_chart[col] = type_chart[col].astype(str).str.strip().str.lower()

    print(f"SUCCESS: cleaned {len(type_chart)} entries.")
    return type_chart

def load_type_chart(file_path: str) -> pd.DataFrame | None:
    """
    Load and normalize a type chart CSV
    """

    print(f"Loading data from '{file_path}'...")
    try:
        raw_df = pd.read_csv(file_path)
        return normalize_type_chart(raw_df)

    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")


# --- MULTPLIERS --- 
def get_multiplier(move_type: str, pkm_type1: str, pkm_type2: str) -> float:
    """
    Return the move's combined effectiveness against the pokemon's types
    """
    
    m1 = type_chart.get((move_type, pkm_type1), 1.0) 
    m2 = type_chart.get((move_type, pkm_type2), 1.0)
    return m1 * m2


# Getting type charts
type_chart = load_type_chart(TYPE_CHART_PATH)
df_indexed = type_chart.set_index(["attacking_type", "defending_type"])
type_chart = df_indexed["multiplier"].to_dict()
stats_map = create_stats_map(train_data, test_data)

## 2. Extract Features (both Train and Test Features)

In [ ]:
# --- BASE STATS ---
def get_stats(turn: dict, p2_stats: dict) -> None:
    """
    Store player 2's active pokemon stats in p2_stats if found in stats_map
    """

    pkm2_name = turn["p2_pokemon_state"]["name"]
    if pkm2_name in stats_map:
        stats = stats_map[pkm2_name]["base_stats"]
        p2_stats[pkm2_name] = stats


def feature_base_stats(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add base pokemon statistics:
    - player 1 full team absolute stats (measures team building quality)
    - player 2 seen team absolute stats (measures active counters deployed)
    - differential stats (P1 - P2: higher = P1 advantage)
    """

    p1_team = battle["p1_team_details"]
    features["p2_stat_coverage"] = len(p2_stats) / TEAM_SIZE
    features["p1_mean_hp"]  = np.mean([p["base_hp"]  for p in p1_team])
    features["p1_mean_atk"] = np.mean([p["base_atk"] for p in p1_team])
    features["p1_mean_def"] = np.mean([p["base_def"] for p in p1_team])
    features["p1_mean_spa"] = np.mean([p["base_spa"] for p in p1_team])
    features["p1_mean_spd"] = np.mean([p["base_spd"] for p in p1_team])
    features["p1_mean_spe"] = np.mean([p["base_spe"] for p in p1_team])
    features["p1_mean_bst"] = features["p1_mean_hp"] + features["p1_mean_atk"] + features["p1_mean_def"] + features["p1_mean_spa"] + features["p1_mean_spd"] + features["p1_mean_spe"]

        # Single values + merged values
        features[f"{player}_sum_bst"] = 0
        features[f"{player}_mean_bst"] = 0
        bst_values = np.zeros(len(team))  # One total per Pokemon with known stats
        for stat in STATISTICS:
            values = [pokemon[f"base_{stat}"] for pokemon in team]
            features[f"{player}_sum_{stat}"]  = np.sum(values)
            features[f"{player}_mean_{stat}"] = np.mean(values)
            features[f"{player}_min_base_{stat}"] = min(values)
            features[f"{player}_max_base_{stat}"] = max(values)
            features[f"{player}_std_base_{stat}"] = float(np.std(values, ddof=0))

            features[f"{player}_sum_bst"] += features[f"{player}_sum_{stat}"]
            bst_values += values

        # Merged values
        features[f"{player}_mean_bst"] = features[f"{player}_sum_bst"] / TEAM_SIZE
        features[f"{player}_std_base_bst"] = float(np.std(bst_values, ddof=0))

    # Differentials
    for stat in STATISTICS:
        features[f"diff_sum_{stat}"]  = features[f"p1_sum_{stat}"] - features[f"p2_sum_{stat}"]
        features[f"diff_mean_{stat}"] = features[f"p1_mean_{stat}"] - features[f"p2_mean_{stat}"]
    features["diff_sum_bst"]  = features["p1_sum_bst"]  - features["p2_sum_bst"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_mean_bst"]
    features["diff_std_base_bst"] = features["p1_std_base_bst"] - features["p2_std_base_bst"]


def feature_team_type_profile(battle: dict, p2_stats: dict, features: dict) -> None:
    """Summarize known team type diversity and repeated defensive weaknesses."""
    p1_team = [pokemon["name"] for pokemon in battle["p1_team_details"]]
    for player, team in (("p1", p1_team), ("p2", p2_stats)):
        team_types = []
        unique_types = set()
        for name in team:
            # If name is present then we collect the types related
            types = [type for type in stats_map.get(name, {}).get("types", [])]
            if types:
                # Ensure first 2 entries
                team_types.append(types)
                unique_types.update(types)

        # Number of pokemons with known types
        known_count = len(team_types)
        features[f"{player}_known_type_count"] = known_count
        features[f"{player}_type_diversity"] = len(unique_types)
        
        total_weakness_severity, shared_weakness_type_count = 0.0, 0
        for attack_type in list({attack_type for attack_type, _ in type_chart}):
            multipliers = [get_multiplier(attack_type, *types) for types in team_types]
            # Sum of (super-)effective moves off team based on attact_type
            total_weakness_severity += sum((multiplier // 2) if multiplier > 1.0 else 0.0 for multiplier in multipliers)
            # At least two pokemon are weak to the same type (multiplier > 1.0)
            shared_weakness_type_count += sum(multiplier > 1.0 for multiplier in multipliers) >= 2

        features[f"{player}_average_weakness_severity"] = total_weakness_severity / known_count if known_count else np.nan
        features[f"{player}_shared_weakness_type_count"] = shared_weakness_type_count if known_count else np.nan

    features["diff_type_diversity"] = features["p1_type_diversity"] - features["p2_type_diversity"]
    features["diff_average_weakness_severity"]  = features["p2_average_weakness_severity"] - features["p1_average_weakness_severity"]
    features["diff_shared_weakness_type_count"] = features["p2_shared_weakness_type_count"] - features["p1_shared_weakness_type_count"]


# --- HP LOSS ---
def get_hp_pct_lost(turn: dict, p1_hp_map: dict, p2_hp_map: dict) -> None:
    """
    Track HP loss from one turn for each observed pokemon:
    - remember its latest HP across switches, starting from full HP
    - accumulate HP loss in its three ten-turn intervals
    Healing reduces HP loss.
    """

    for player in ("p1", "p2"):
        name = turn[f"{player}_pokemon_state"]["name"]
        hp = turn[f"{player}_pokemon_state"]["hp_pct"]
        
        # Initialize the HP intervals list for player's pokemon if not already present
        if player == "p1":
            if name not in p1_hp_map:
                p1_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
            pokemon = p1_hp_map[name]
        else: # p2
            if name not in p2_hp_map:
                p2_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
            pokemon = p2_hp_map[name]

        interval_index = (turn["turn"] - 1) // 10 # [0-> 1-10, 1-> 11-20, 2-> 21-30]
        pokemon["losses"][interval_index] += pokemon["last_hp"] - hp
        pokemon["last_hp"] = hp

def feature_hp_lost(p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add net HP loss in each ten-turn interval and in total, divided by TEAM_SIZE.
    Unseen pokemon contribute zero loss; healing reduces loss.
    Differentials use P2 loss - P1 loss: higher = P1 advantage.
    """

    for player in ("p1", "p2"):
        if player == "p1":
            team = p1_hp_map
        else:
            team = p2_hp_map

        # Single values + merged values
        features[f"{player}_sum_hp_lost"] = 0
        # Keep one total per Pokemon, with zero loss for unseen Pokemon.
        hp_lost_values = np.zeros(TEAM_SIZE)
        for i, interval in enumerate(HP_PCT_LOST_INTERVALS):
            values = [pokemon["losses"][i] for pokemon in team.values()]
            features[f"{player}_hp_pct_lost_{interval}"] = sum(values)
            features[f"{player}_sum_hp_lost"] += features[f"{player}_hp_pct_lost_{interval}"]
            hp_lost_values[:len(team)] += values

        # Merged values
        features[f"{player}_mean_hp_pct_lost"] = features[f"{player}_sum_hp_lost"] / TEAM_SIZE
        features[f"{player}_std_hp_pct_lost"] = float(np.std(hp_lost_values, ddof=0))

    # Differentials
    for interval in HP_PCT_LOST_INTERVALS:
        features[f"diff_hp_pct_lost_{interval}"] = features[f"p2_hp_pct_lost_{interval}"] - features[f"p1_hp_pct_lost_{interval}"]
    features["diff_sum_hp_pct_lost"]  = features["p2_sum_hp_lost"] - features["p1_sum_hp_lost"]
    features["diff_mean_hp_pct_lost"] = features["p2_mean_hp_pct_lost"] - features["p1_mean_hp_pct_lost"]
    features["diff_std_hp_pct_lost"]  = features["p2_std_hp_pct_lost"] - features["p1_std_hp_pct_lost"]


# --- TYPE EFFECTIVENESS ---
def get_mults(turn: dict, p1_mults: dict, p2_mults: dict) -> None:
    """
    Update both players' type effectiveness multiplier counts for one turn (damaging moves only).
    """

    for player in ("p1", "p2"):
        mv = turn[f"{player}_move_details"]
        if mv is None or mv["category"] == "status":
            continue

        mv_type = mv["type"]
        
        # Types = stats_map["pokemon_name"]["types"]
        pkm_types = stats_map.get(turn[f"{player}_pokemon_state"]["name"], {}).get("types", ["notype", "notype"])
        mult = get_multiplier(mv_type, *pkm_types) # mv_type, types[0], types[1]
        if player == "p1":
            p1_mults[mult] += 1
        else: # p2
            p2_mults[mult] += 1
    

def feature_mults(p1_mults: dict, p2_mults: dict, features: dict) -> None:
    """
    Add bucket counts and population summaries of recorded attack multipliers.
    Count advantage is P1 - P2 above 1x and P2 - P1 below 1x; 1x is neutral.
    """

    for player in ("p1", "p2"):
        if player == "p1":
            team = p1_mults
        else:
            team = p2_mults

        # Single values + merged values
        values = []
        features[f"{player}_count_mult"] = 0
        features[f"{player}_sum_mult"] = 0
        for mult in MULTIPLIERS:
            value = team[mult]
            values.extend([mult] * value)
            features[f"{player}_{mult}_mult"] = value
            features[f"{player}_sum_mult"] += mult * value
            features[f"{player}_count_mult"] += value

        # Merged values
        features[f"{player}_min_mult"] = min(values) if values else np.nan
        features[f"{player}_max_mult"] = max(values) if values else np.nan
        features[f"{player}_mean_mult"] = features[f"{player}_sum_mult"] / features[f"{player}_count_mult"] if values else np.nan
        features[f"{player}_std_mult"] = float(np.std(values, ddof=0)) if values else np.nan

    # Differentials
    features["diff_count_mult"] = 0
    for mult in MULTIPLIERS:
        if mult < 1:
            features[f"diff_{mult}_mult"] = features[f"p2_{mult}_mult"] - features[f"p1_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
        else:
            features[f"diff_{mult}_mult"] = features[f"p1_{mult}_mult"] - features[f"p2_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
    features["diff_min_mult"] = features["p1_min_mult"] - features["p2_min_mult"]
    features["diff_max_mult"] = features["p1_max_mult"] - features["p2_max_mult"]
    features["diff_sum_mult"]  = features["p1_sum_mult"] - features["p2_sum_mult"]
    features["diff_mean_mult"] = features["p1_mean_mult"] - features["p2_mean_mult"]
    features["diff_std_mult"] = features["p1_std_mult"] - features["p2_std_mult"]


# --- STATUSES (ALSO FAINT) ---
def get_statuses(turn: dict, p1_status: dict, p2_status: dict) -> None:
    """
    Accumulate active status turn counts for both players, including fainting
    """

    for player in ("p1", "p2"):
        pkm_state = turn[f"{player}_pokemon_state"]["status"]
        if pkm_state == "nostatus":
            continue

        if player == "p1":
            p1_status[pkm_state] = p1_status.get(pkm_state, 0) + 1
        else: # p2
            p2_status[pkm_state] = p2_status.get(pkm_state, 0) + 1
   

def feature_statuses(p1_status_map: dict, p2_status_map: dict, n_turns: int, features: dict) -> None:
    """
    Add status turn totals, weighted sums, and differentials.
    Pre-initializes all known statuses to eliminate train/test schema mismatches.
    """

    features["p1_statuses_wsum"] = 0
    features["p2_statuses_wsum"] = 0

    # Pre-initialize all statuses from STATUSES_WEIGHTS
    for status, weight in STATUSES_WEIGHTS.items():
        val_sts_1 = p1_status_map.get(status, 0)
        val_sts_2 = p2_status_map.get(status, 0)
        features[f"p1_{status}"] = val_sts_1
        features[f"p2_{status}"] = val_sts_2
        features[f"diff_{status}"] = val_sts_2 - val_sts_1
        features[f"p1_status_frac_{status}"] = val_sts_1 / n_turns
        features[f"p2_status_frac_{status}"] = val_sts_2 / n_turns
        features[f"diff_status_frac_{status}"] = (val_sts_2 - val_sts_1) / n_turns
        features["p1_statuses_wsum"] += val_sts_1 * weight
        features["p2_statuses_wsum"] += val_sts_2 * weight

    features["p1_statuses_wmean"] = features["p1_statuses_wsum"] / n_turns
    features["p2_statuses_wmean"] = features["p2_statuses_wsum"] / n_turns

    # Differentials: P2 minus P1 (higher = P2 suffered more status afflictions = P1 advantage)
    features["diff_statuses_wsum"]  = features["p2_statuses_wsum"] - features["p1_statuses_wsum"]
    features["diff_statuses_wmean"] = features["p2_statuses_wmean"] - features["p1_statuses_wmean"]


# --- BOOSTS (CONSISTENT SCHEMA & DIFFERENTIALS) ---
def get_boosts(turn: dict, p1_boosts: dict, p2_boosts: dict) -> None:
    """
    Accumulate signed boost stages per stat for both active pokemon.
    Update both players' totals in place using the current observed turn
    """

    for player in ("p1", "p2"):
        pkm_boosts: dict = turn[f"{player}_pokemon_state"]["boosts"]
        for stat, value in pkm_boosts.items():
            if player == "p1":
                p1_boosts[stat] += value
            else: # p2
                p2_boosts[stat] += value


def feature_boosts(p1_boosts: dict, p2_boosts: dict, n_turns: int, features: dict) -> None:
    """
    Add signed boost stage totals per stat for both players, weighted sums,
    and differentials favoring P1.
    Pre-initializes all stats to eliminate train/test schema mismatches.
    """

    features["p1_boosts_wsum"] = 0
    features["p2_boosts_wsum"] = 0

    # Pre-initialize all boosts from BOOSTS_WEIGHTS
    for boost, weight in BOOSTS_WEIGHTS.items():
        val_bst_1 = p1_boosts.get(boost, 0)
        val_bst_2 = p2_boosts.get(boost, 0)
        features[f"p1_{boost}_boost_mean"] = val_bst_1 / n_turns
        features[f"p2_{boost}_boost_mean"] = val_bst_2 / n_turns
        features[f"diff_{boost}_boost_mean"] = (val_bst_1 - val_bst_2) / n_turns
        features[f"p1_{boost}_boost"] = val_bst_1
        features[f"p2_{boost}_boost"] = val_bst_2
        features[f"diff_{boost}_boost"] = val_bst_1 - val_bst_2
        features["p1_boosts_wsum"] += val_bst_1 * weight
        features["p2_boosts_wsum"] += val_bst_2 * weight

    features["p1_boosts_wmean"] = features["p1_boosts_wsum"] / n_turns
    features["p2_boosts_wmean"] = features["p2_boosts_wsum"] / n_turns

    # Differentials (P1 - P2: higher = P1 has more positive stat stages)
    features["diff_boosts_wsum"]  = features["p1_boosts_wsum"] - features["p2_boosts_wsum"]
    features["diff_boosts_wmean"] = features["p1_boosts_wmean"] - features["p2_boosts_wmean"]


# --- EFFECTS FEATURES ---
def get_effects(turn: dict, p1_effects: dict, p2_effects: dict) -> None:
    """
    Accumulate positive and negative effect weights from one turn for both players
    """

    for player in ("p1", "p2"):
        for eff in turn[f"{player}_pokemon_state"]["effects"]:
            w = EFFECT_WEIGHTS.get(eff, 0.0)
            label = "pos" if w > 0 else "neg" 
            if player == "p1":
                p1_effects[label] += w
            else: # p2
                p2_effects[label] += w

def feature_effects(p1_effects: dict, p2_effects: dict, n_turns: int, features: dict) -> None:
    """
    Add weighted effect scores after all observed turns and at the final turn:
    - mean positive, negative, and net effect weights for P1 and P2
    - net weighted effect difference (P1 - P2) across the battle
    """

    features["p1_eff_pos_mean"] = p1_effects["pos"] / n_turns
    features["p1_eff_neg_mean"] = p1_effects["neg"] / n_turns
    features["p2_eff_pos_mean"] = p2_effects["pos"] / n_turns
    features["p2_eff_neg_mean"] = p2_effects["neg"] / n_turns

    features["p1_eff_net_mean"]   = features["p1_eff_pos_mean"] + features["p1_eff_neg_mean"]
    features["p2_eff_net_mean"]   = features["p2_eff_pos_mean"] + features["p2_eff_neg_mean"]

    features["diff_pos_mean"]     = features["p1_eff_pos_mean"] - features["p2_eff_pos_mean"]
    features["diff_neg_mean"]     = features["p2_eff_neg_mean"] - features["p1_eff_neg_mean"]
    features["diff_eff_net_mean"] = features["p1_eff_net_mean"] - features["p2_eff_net_mean"]


# --- TEAM SNAPSHOT AT LAST TURN ---
def get_last_seen(turn: dict, p1_last_seen: dict, p2_last_seen: dict) -> None:
    """
    Store each observed pokemon's latest HP and status stages.
    Update both players' maps in place; retain switched-out pokemon's snapshots.
    """

    for player in ("p1", "p2"):
        pkm_state = turn[f"{player}_pokemon_state"]
        name = pkm_state["name"]
        if player == "p1":
            p1_last_seen[name] = {
                "hp_pct": pkm_state["hp_pct"],
                "status": pkm_state["status"]
            }
        else: # p2
            p2_last_seen[name] = {
                "hp_pct": pkm_state["hp_pct"],
                "status": pkm_state["status"]
            }


def get_p2_faint_evidence(turn: dict, idx: int, timeline: list, tracker: dict) -> None:
    """Accumulate deterministic and likely Player 2 KO evidence for one turn."""
    p2 = turn["p2_pokemon_state"]
    p1 = turn["p1_pokemon_state"]
    p1_move = turn.get("p1_move_details") or {}
    p2_move = turn.get("p2_move_details") or {}
    
    name      = p2["name"]
    hp        = p2["hp_pct"]
    status    = p2["status"]
    effects   = p2["effects"]
    move_name = p2_move.get("name", "")

    tracker["spikes_active"] = p1_move.get("name", "") == "spikes"
    if (status == "fnt" or hp <= 0) or (move_name in {"explosion", "selfdestruct"}) or ("perish" in effects and hp <= 0):
        tracker["confirmed"].add(name)

    # The next active Pokemon identifies a departure in the recorded timeline.
    if (
        idx + 1 >= len(timeline) 
        or name in tracker["confirmed"] 
        or any(t["p2_pokemon_state"]["name"] == name for t in timeline[idx + 1:])
        or move_name in {"roar", "whirlwind"}
    ):
        return
    
    damaging     = p1_move.get("base_power", 0) > 0 and p1_move.get("category") in {"physical", "special"}
    residual     = hp <= 0.15 and (status in {"tox", "psn", "brn"} or "leechseed" in effects or tracker["spikes_active"])
    destiny_bond = "destinybond" in effects or "destinybond" in p1["effects"]
    if residual or (destiny_bond and damaging) or (damaging and hp <= 0.40):
        tracker["ghost"].add(name)


def feature_p2_faint_evidence(tracker: dict, p2_last_seen: dict, features: dict) -> None:
    """Expose reconstructed KO counts, true differential, and Player 2 remaining bulk."""
    confirmed = tracker["confirmed"]
    features["p2_confirmed_kos"] = len(confirmed)
    features["p2_estimated_true_kos"] = len(confirmed | tracker["ghost"])
    
    # Grounded differential: Reconstructed P2 casualties - P1 casualties
    features["diff_estimated_pokemon_alive"] = features["p2_estimated_true_kos"] - features.get("p1_fnt", 0)
    
    known_hp = sum(float(p.get("hp_pct", 1.0) or 0.0) for p in p2_last_seen.values())
    features["p2_known_remaining_hp_sum"] = known_hp + max(0, TEAM_SIZE - len(p2_last_seen))

def feature_last_seen(last_turn: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add latest-known team HP, status, boost, and effect means and P1 - P2 differences.
    Divide by six; assume unseen pokemon have full HP and no status, effects, or boosts.
    Benched pokemon retain their last observed data, which may be older.
    """

    features["p1_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p1_last_seen))
    features["p1_lst_statuses_wmean"] = 0.0
    features["p1_lst_effects_wmean"] = 0.0
    features["p1_lst_boosts_wmean"] = 0.0
    for pkm1 in p1_last_seen.values():
        features["p1_lst_hp_mean"] += pkm1["hp_pct"]
        features["p1_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm1["status"], 0)
    lst_pkm1_effects = last_turn["p1_pokemon_state"]["effects"]
    for eff in lst_pkm1_effects:
        features["p1_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm1_boosts = last_turn["p1_pokemon_state"]["boosts"]
    for bst, value in lst_pkm1_boosts.items():
        features["p1_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p1_lst_hp_mean"]        /= TEAM_SIZE
    features["p1_lst_statuses_wmean"] /= TEAM_SIZE
    features["p1_lst_effects_wmean"]  /= TEAM_SIZE
    features["p1_lst_boosts_wmean"]   /= TEAM_SIZE

    features["p2_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p2_last_seen))
    features["p2_lst_statuses_wmean"] = 0.0
    features["p2_lst_effects_wmean"] = 0.0
    features["p2_lst_boosts_wmean"] = 0.0
    for pkm2 in p2_last_seen.values():
        features["p2_lst_hp_mean"] += pkm2["hp_pct"]
        features["p2_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm2["status"], 0)
    lst_pkm2_effects = last_turn["p2_pokemon_state"]["effects"]
    for eff in lst_pkm2_effects:
        features["p2_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm2_boosts = last_turn["p2_pokemon_state"]["boosts"]
    for bst, value in lst_pkm2_boosts.items():
        features["p2_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p2_lst_hp_mean"]        /= TEAM_SIZE
    features["p2_lst_statuses_wmean"] /= TEAM_SIZE
    features["p2_lst_effects_wmean"]  /= TEAM_SIZE
    features["p2_lst_boosts_wmean"]   /= TEAM_SIZE

    features["diff_lst_hp_mean"]        = features["p1_lst_hp_mean"] - features["p2_lst_hp_mean"]
    features["diff_lst_statuses_wmean"] = features["p2_lst_statuses_wmean"] - features["p1_lst_statuses_wmean"]
    features["diff_lst_effects_wmean"]  = features["p1_lst_effects_wmean"] - features["p2_lst_effects_wmean"]
    features["diff_lst_boosts_wmean"]   = features["p1_lst_boosts_wmean"] - features["p2_lst_boosts_wmean"]


# --- HELD ITEMS ---
def get_items(turn: dict, p2_item_tracker: dict) -> None:
    """
    Track observed P2 item activations turn by turn:
    - detects Leftovers recovery (approx 1/16 = 6.25% HP tick without healing moves)
    - detects item signatures appearing in turn effects
    """
    # current player 2 pokemon state
    pkm_state   = turn["p2_pokemon_state"]
    pkm_name    = pkm_state["name"]
    curr_hp     = pkm_state["hp_pct"]
    pkm_effects = pkm_state["effects"]

    # current player 2 move details
    mv = turn["p2_move_details"]
    mv_name = mv["name"] if mv else ""

    # Check Leftovers passive regeneration between consecutive turns of the same active Pokemon
    if pkm_name == p2_item_tracker.get("prev_mon"):
        hp_diff = curr_hp - p2_item_tracker.get("prev_hp")
        if hp_diff > 0 and mv_name not in HEALING_MOVES:
            p2_item_tracker["leftovers_seen"].add(pkm_name)

    # Check item names explicitly exposed in turn effects
    for eff in pkm_effects:
        if eff in JUNK_ITEMS:
            p2_item_tracker["junk_seen"].add((pkm_name, eff))
        elif eff in RNG_ITEMS:
            p2_item_tracker["rng_seen"].add((pkm_name, eff))

    p2_item_tracker["prev_mon"] = pkm_name
    p2_item_tracker["prev_hp"] = curr_hp

def feature_items(battle: dict, p2_item_tracker: dict, features: dict) -> None:
    """
    Extract Player 1 item stats, Player 2 timeline detections, and their differentials
    """

    p1_team = battle.get("p1_team_details", [])
    p1_items = [p.get("item") or "noitem" for p in p1_team]

    # 1. Macro team item metrics
    p1_leftovers = p1_items.count("leftovers")
    features["p1_item_leftovers_count"]   = p1_leftovers
    features["p1_item_has_all_leftovers"] = int(p1_leftovers == 6)
    features["p1_item_unique_count"]      = len(set(p1_items))
    features["p1_item_non_leftovers"]     = 6 - p1_leftovers

    # 2. Archetype groupings
    p1_junk = sum(1 for it in p1_items if it in JUNK_ITEMS)
    features["p1_item_junk_count"]         = p1_junk
    features["p1_item_status_berry_count"] = sum(1 for it in p1_items if it in STATUS_BERRIES)
    features["p1_item_type_boost_count"]   = sum(1 for it in p1_items if it in TYPE_BOOST_ITEMS)
    features["p1_item_rng_count"]          = sum(1 for it in p1_items if it in RNG_ITEMS)

    # 3. High-impact item synergies
    features["p1_has_thickclub_marowak"] = int(any(
        p.get("name") in {"marowak", "cubone"} and p.get("item") == "thickclub"
        for p in p1_team
    ))
    features["p1_has_lightball_pikachu"] = int(any(
        p.get("name") == "pikachu" and p.get("item") == "lightball"
        for p in p1_team
    ))

    # 4. Player 2 timeline detections and differentials
    p2_leftovers = len(p2_item_tracker["leftovers_seen"])
    p2_junk = len(p2_item_tracker["junk_seen"])

    features["p2_detected_leftovers_count"] = p2_leftovers
    features["p2_detected_rng_count"] = len({name for name, _ in p2_item_tracker["rng_seen"]})
    features["diff_leftovers_count"]        = p1_leftovers - p2_leftovers
    features["diff_junk_items_count"]       = p1_junk - p2_junk


# --- DEPLOYED LEVELS & EFFECTIVE HP (OUTLIER-AWARE & BASE HP x LEVEL x HP% CORRELATION) ---
def calc_eff_stat(base_val: float, level: int, is_hp: bool = False) -> float:
    """
    Compute the Gen 2 effective stat from base stat and level (assuming standard max DV=15 and max StatExp).
    Correlates base stats with Pokemon level directly.
    """

    core = ((base_val + 47) * 2 * level) / 100.0
    return core + level + 10 if is_hp else core + 5


def get_deployed_turns(turn: dict, p1_deployed_turns: dict, p2_deployed_turns: dict) -> None:
    """
    Count active turns per observed Pokemon for both players
    """

    for player, team in (("p1", p1_deployed_turns), ("p2", p2_deployed_turns)):
        name = turn[f"{player}_pokemon_state"]["name"]
        team[name] = team.get(name, 0) + 1

def feature_deployed_and_levels(battle: dict, p1_deployed_turns: dict, p2_deployed_turns: dict, p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add deployment counts, level summaries and estimated remaining HP ratios.
    P1 uses supplied team levels; P2 uses its known lead and assumes level 100 otherwise.
    Unseen P2 Pokemon have full HP and assumed base HP 85; unknown seen species use 80.
    Differences use P1 - P2. Deployment count has no fixed good/bad meaning
    """

    p1_team = {pokemon["name"]: pokemon for pokemon in battle["p1_team_details"]}
    p1_lead = next(iter(p1_deployed_turns), None)
    p2_lead = battle["p2_lead_details"]
    level_sum, weighted_level_sum = 0, 0
    min_level, deployed_min_level = np.inf, np.inf
    underleveled_count, deployed_underleveled_count = 0, 0
    hp_ratios = {}

    for player, team, deployed_turns, hp_map in (
        ("p1", p1_team, p1_deployed_turns, p1_hp_map),
        ("p2", p2_deployed_turns, p2_deployed_turns, p2_hp_map),
    ):
        remaining_hp, total_hp = 0.0, 0.0
        for name in team:
            if player == "p1":
                pokemon = p1_team[name]
                level = pokemon["level"]
                level_sum += level
                min_level = min(min_level, level)
                underleveled_count += level < 100
                if name in deployed_turns:
                    deployed_min_level = min(deployed_min_level, level)
                    deployed_underleveled_count += level < 100
                    weighted_level_sum += level * deployed_turns[name]
            elif name == p2_lead["name"]:
                pokemon = p2_lead
            else:
                pokemon = {
                    "base_hp": stats_map.get(name, {}).get("base_stats", {}).get("base_hp", 80),
                    "level": 100,
                }
            hp = calc_eff_stat(pokemon["base_hp"], pokemon["level"], is_hp=True)
            total_hp += hp
            remaining_hp += hp * hp_map.get(name, {}).get("last_hp", 1.0)

        if player == "p2":
            unseen_hp = (TEAM_SIZE - len(team)) * calc_eff_stat(85, 100, is_hp=True)
            total_hp += unseen_hp
            remaining_hp += unseen_hp
        hp_ratios[player] = remaining_hp / total_hp if total_hp else 0.0

    features["p1_deployed_count"] = len(p1_deployed_turns)
    features["p2_deployed_count"] = len(p2_deployed_turns)
    features["diff_deployed_count"] = features["p1_deployed_count"] - features["p2_deployed_count"]
    features["p1_min_level"] = min_level
    features["p1_mean_level"] = level_sum / len(p1_team)
    features["p1_underleveled_count"] = underleveled_count
    features["p1_deployed_underleveled_count"] = deployed_underleveled_count
    features["p1_deployed_min_lvl"] = deployed_min_level if deployed_min_level != np.inf else 100.0
    features["p1_active_wmean_lvl"] = weighted_level_sum / max(sum(p1_deployed_turns.values()), 1)
    # The first observed Pokemon is the lead, regardless of the team list order.
    features["p1_lead_level"] = p1_team.get(p1_lead, {}).get("level", np.nan)
    features["p2_lead_level"] = p2_lead["level"]
    features["diff_lead_level"] = features["p1_lead_level"] - features["p2_lead_level"]
    features["p1_rem_effective_hp_ratio"] = hp_ratios["p1"]
    features["p2_rem_effective_hp_ratio"] = hp_ratios["p2"]
    features["diff_rem_effective_hp_ratio"] = features["p1_rem_effective_hp_ratio"] - features["p2_rem_effective_hp_ratio"]


# --- TACTICAL TEMPO, OFFENSIVE POWER, HAZARDS & REST ---
def get_tactics(turn: dict, p1_tactics: dict, p2_tactics: dict) -> None:
    """
    Track turn-by-turn tactical momentum for both players:
    - voluntary switches (excluding post-KO replacements)
    - distinct fainted Pokemon
    - effective offensive power (Base Power * STAB * Type Multiplier * Accuracy)
    - Rest / Sleep Talk usage and Spikes / Phazing (Roar/Whirlwind)
    """
    for side, opp_side, tracker, opp_tracker in (
        ("p1", "p2", p1_tactics, p2_tactics),
        ("p2", "p1", p2_tactics, p1_tactics),
    ):
        state      = turn[f"{side}_pokemon_state"]
        opp_state  = turn[f"{opp_side}_pokemon_state"]
        mv         = turn[f"{side}_move_details"]
        name       = state["name"]
        is_fainted = (state["status"] == "fnt" or state["hp_pct"] == 0.0)

        # Conta solo i cambi volontari (esclude gli ingressi forzati dopo un KO al turno precedente)
        prev_pkm = tracker.get("prev_mon")
        if (prev_pkm and name != prev_pkm) and not tracker.get("prev_fainted", False):
            tracker["switches"] += 1
        
        tracker["prev_mon"] = name
        tracker["prev_fainted"] = is_fainted

        # Track distinct fainted Pokemon
        if is_fainted:
            tracker["fainted_mons"].add(name)

        # Track offensive moves and their effective power
        if mv is None:
            continue

        tracker["moves_used"] += 1
        mv_name       = mv["name"]
        mv_type       = mv["type"]
        mv_category   = mv["category"]
        mv_base_power = mv["base_power"]

        # Effective offensive power for damaging moves
        if mv_category != "status" and mv_base_power > 0:
            attacker_types = stats_map.get(name, {}).get("types", ["notype", "notype"])
            defender_types = stats_map.get(opp_state["name"], {}).get("types", ["notype", "notype"])
            
            # If the attacker uses a move that matches one of its types, apply STAB (Same Type Attack Bonus)
            stab = STAB if mv_type in attacker_types else 1.0
            tracker["power_sum"] += mv["base_power"] * stab * mv["accuracy"] * get_multiplier(mv_type, *defender_types)

        # Azioni di stato valide solo se il Pokemon non è andato KO prima di eseguirle
        if is_fainted:
            continue

        match mv_name:
            case "rest":
                tracker["rest_used"] += 1
            case "sleeptalk":
                tracker["sleeptalk_used"] += 1
            case "spikes":
                tracker["spikes_active"] = 1
            case "rapidspin":
                opp_tracker["spikes_active"] = 0
            case "roar" | "whirlwind":
                tracker["phaze_used"] += 1

def feature_tactics(p1_tactics: dict, p2_tactics: dict, n_turns: int, features: dict) -> None:
    """Compute tactical differentials from accumulated timeline trackers."""
    # 1. Action tempo & switching
    features["p1_moves_used"]   = p1_tactics["moves_used"] / n_turns
    features["p2_moves_used"]   = p2_tactics["moves_used"] / n_turns
    features["diff_moves_used"] = (p1_tactics["moves_used"] - p2_tactics["moves_used"]) / n_turns
    features["p1_switches"]     = p1_tactics["switches"] / n_turns
    features["p2_switches"]     = p2_tactics["switches"] / n_turns
    features["diff_switches"]   = (p1_tactics["switches"] - p2_tactics["switches"]) / n_turns

    # 2. Effective offensive power per turn
    features["p1_eff_power_mean"]   = p1_tactics["power_sum"] / n_turns
    features["p2_eff_power_mean"]   = p2_tactics["power_sum"] / n_turns
    features["diff_eff_power_mean"] = features["p1_eff_power_mean"] - features["p2_eff_power_mean"]

    # 3. Gen 2 Meta: RestTalk & Spikes/Phazing
    features["p1_rest_used"]        = p1_tactics["rest_used"]
    features["p2_rest_used"]        = p2_tactics["rest_used"]
    features["diff_rest_used"]      = p1_tactics["rest_used"] - p2_tactics["rest_used"]
    features["p1_sleeptalk_used"]   = p1_tactics["sleeptalk_used"]
    features["p2_sleeptalk_used"]   = p2_tactics["sleeptalk_used"]
    features["diff_sleeptalk_used"] = p1_tactics["sleeptalk_used"] - p2_tactics["sleeptalk_used"]
    features["p1_spikes_active"]    = p1_tactics["spikes_active"]
    features["p2_spikes_active"]    = p2_tactics["spikes_active"]
    features["diff_spikes_active"]  = p1_tactics["spikes_active"] - p2_tactics["spikes_active"]
    features["p1_phaze_used"]       = p1_tactics["phaze_used"]
    features["p2_phaze_used"]       = p2_tactics["phaze_used"]
    features["diff_phaze_used"]     = p1_tactics["phaze_used"] - p2_tactics["phaze_used"]

In [ ]:
# --- EXTRACT FEATURES ---
def extract_features(data: list) -> pd.DataFrame:
    """
    Extract comprehensive features combining absolute scale and differentials:
    1. base stats (P1 absolute, P2 absolute, diff P1 - P2)
    2. HP loss in intervals and total (P1, P2, and differentials)
    3. type effectiveness multiplier counts and differentials
    4. statuses (pre-initialized, consistent schema, and differentials)
    5. boosts (P1, P2, and differentials)
    6. battle effects (P1, P2, and differentials)
    7. last-turn snapshot at Turn 30 (P1, P2, and differentials)
    8. held items (P1 known items, P2 timeline detections, and differentials)
    9. team type profile and defensive matchup summaries
    10. deployed Pokemon levels, symmetry counts & effective HP ratios
    11. tactical tempo, effective offensive power, hazards, phazing & RestTalk
    """

    print("Extracting features...")
    feature_list = []

    for battle in data:
        features = {"battle_id" : battle["battle_id"]}

        # Keep the identifier and target for the training and submission cells
        if "player_won" in battle:
            features["player_won"] = battle["player_won"]


        # Initializations of data structures
        p1_hp_map, p2_hp_map = {}, {}
        p1_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p2_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p1_status, p2_status = {}, {}
        p1_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p2_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p1_effects = {"pos": 0.0, "neg": 0.0}
        p2_effects = {"pos": 0.0, "neg": 0.0}
        p1_last_seen, p2_last_seen = {}, {}
        p2_stats = {}
        p2_item_tracker = {
            "leftovers_seen": set(),
            "junk_seen": set(),
            "rng_seen": set(),
            "prev_mon": None,
            "prev_hp": None,
        }
        p1_deployed_turns, p2_deployed_turns = {}, {}
        p2_faint_tracker = {"confirmed": set(), "ghost": set(), "spikes_active": False}
        p1_tactics = {
            "prev_mon": None, "prev_fainted": False, "switches": 0, "fainted_mons": set(), "moves_used": 0,
            "rest_used": 0, "sleeptalk_used": 0, "spikes_active": 0, "phaze_used": 0, "power_sum": 0.0
        }
        p2_tactics = {
            "prev_mon": None, "prev_fainted": False, "switches": 0, "fainted_mons": set(), "moves_used": 0,
            "rest_used": 0, "sleeptalk_used": 0, "spikes_active": 0, "phaze_used": 0, "power_sum": 0.0
        }

        timeline = battle["battle_timeline"]
        for turn_idx, turn in enumerate(timeline):
            get_stats(turn, p2_stats)
            get_hp_pct_lost(turn, p1_hp_map, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)
            get_statuses(turn, p1_status, p2_status)
            get_boosts(turn, p1_boosts, p2_boosts)
            get_effects(turn, p1_effects, p2_effects)
            get_last_seen(turn, p1_last_seen, p2_last_seen)
            get_items(turn, p2_item_tracker)
            get_deployed_turns(turn, p1_deployed_turns, p2_deployed_turns)
            get_tactics(turn, p1_tactics, p2_tactics)
            get_p2_faint_evidence(turn, turn_idx, timeline, p2_faint_tracker)

        last_turn = timeline[-1]
        # 1. Base stats
        feature_base_stats(battle, p2_stats, features)
        feature_team_type_profile(battle, p2_stats, features)
        # 2. HP loss in intervals, totals & differentials
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 3. Type effectiveness multiplier counts & differentials
        feature_mults(p1_mults, p2_mults, features)
        # 4. Statuses & differentials
        # feature_statuses(p1_status, p2_status, len(timeline), features)
        # 5. Boosts & differentials
        # feature_boosts(p1_boosts, p2_boosts, len(timeline), features)
        # 6. Effects & differentials
        # feature_effects(p1_effects, p2_effects, len(timeline), features)
        # 7. Last seen & differentials
        feature_last_seen(last_turn, p1_last_seen, p2_last_seen, features)
        feature_p2_faint_evidence(p2_faint_tracker, p2_last_seen, features)
        # 8. Items
        # feature_items(battle, p2_item_tracker, features)
        # 9. Deployed Pokemon levels & effective stats (Base Stats x Level)
        # feature_deployed_and_levels(battle, p1_deployed_turns, p2_deployed_turns, p1_hp_map, p2_hp_map, features)
        # 10. Tactical tempo, offensive power, hazards & RestTalk
        # feature_tactics(p1_tactics, p2_tactics, len(timeline), features)

        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

## 3. Training

### Models

In [ ]:
def build_model( random_state: int = RANDOM_STATE, iterations: int | None = None ) -> CatBoostClassifier:
    """
    Build an instance of logistic_regression with seed `random_state`
    """

    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=2.0, random_state=random_state)
    )

def make_lgbm(random_state: int) -> LGBMClassifier:
    """
    Build an instance of `lightgbm` with seed `random_state`
    """
        
    return make_pipeline(
        StandardScaler(),
        LGBMClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        verbose=-1,
        random_state=random_state
        )
    )

def prepare_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple:
    """
    Build an instance of `xgboost` with seed `random_state`
    """
    return make_pipeline(
        StandardScaler(),
        XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1
        )
    )

def make_catboost(random_state: int):
    """Build CatBoost using symmetric (oblivious) trees."""
    return make_pipeline(
        StandardScaler(),
        CatBoostClassifier(
            iterations=150,
            depth=5,
            learning_rate=0.04,
            loss_function="Logloss",
            verbose=False,
            allow_writing_files=False,
            random_seed=random_state,
            thread_count=1
        )
    )

def build_model(model_type: str = MODEL_TYPE, random_state: int = RANDOM_STATE): # -> BaseEstimator:
    """
    Build an instance of `model_type` with seed `random_state`
    """

    if USE_FEATURE_SELECTION:
        feature_cols = list(dict.fromkeys(ALL_FEATURES))
    elif USE_ALLOWED_FEATURES:
        feature_cols = list(dict.fromkeys(ALLOWED_FEATURES))
    else:
        feature_cols = [
            col for col in train_df.columns
            if col not in {"battle_id", "player_won"}
        ]
    if not feature_cols:
        raise ValueError("No predictors selected. Check extraction and the configured feature lists.")
    if {"battle_id", "player_won"}.intersection(feature_cols):
        raise ValueError("Identifiers and target labels cannot be predictors.")
    for name, frame in (("training", train_df), ("test", test_df)):
        missing = sorted(set(feature_cols) - set(frame.columns))
        if missing:
            message = f"Missing {name} features: {missing}"
            if USE_FEATURE_SELECTION:
                message += (
                    ". Feature selection requires every entry in ALL_FEATURES. "
                    "Enable the relevant feature extraction calls and rerun extraction, "
                    "or explicitly restrict ALL_FEATURES to the intended candidate pool."
                )
            raise ValueError(message)
    unknown_categorical = sorted(set(CATEGORICAL_FEATURES) - set(feature_cols))
    if unknown_categorical:
        raise ValueError(f"Categorical columns must be selected: {unknown_categorical}")

        case "lightgbm":
            return make_lgbm(random_state)

        case "xgboost":
            return make_xgb(random_state)

        case "catboost":
            return make_catboost(random_state)

        case "ensemble":
            return VotingClassifier(
                estimators=[
                    ("lgb", make_lgbm(random_state)),
                    ("xgb", make_xgb(random_state)),
                    ("cat", make_catboost(random_state)),
                    ("lr", make_lr(random_state)),
                ],
                voting="soft",
                weights=[1.0, 1.0, 1.0, 1.0],
            )

        case _:
            raise ValueError(f"ERROR: Unavailable model: {model_type}.")

### Performance metrics

In [ ]:
# --- METRICS ---
def compute_metrics(y_true: pd.Series | np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    """
    Return accuracy, precision, recall, F1, and ROC-AUC for binary predictions.
    y_true and y_pred contain 0/1 labels.
    y_prob contains probabilities of class 1
    """

    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) = 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        # Area Under the Receiver Operating Characteristic Curve
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

### Training

In [ ]:
# --- TRAINING ---
def train_and_evaluate_cv(
    X: pd.DataFrame,
    y: pd.Series,
    X_test: pd.DataFrame,
    model_type: str = MODEL_TYPE,
    k: int = K_FOLDS,
    n_repeats: int = N_REPEATS,
    random_state: int = RANDOM_STATE,
):
    """
    Repeated Stratified K-Fold Cross-Validation:
    1. Evaluates training and validation metrics for standalone models or ensembles.
    2. Dynamically detects single estimators vs. VotingClassifier to prevent attribute errors.
    3. Computes out-of-fold (OOF) probabilities and fold-averaged test predictions.
    """
    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=random_state)
    n_total = k * n_repeats

    # Dynamic model naming based on whether MODEL_TYPE is an ensemble or standalone
    if model_type == "ensemble":
        model_names = ["ensemble", "lgb", "xgb", "cat", "lr"]
    else:
        model_names = [model_type]

    display_names = {
        "ensemble": "Ensemble",
        "lgb": "LightGBM",
        "xgb": "XGBoost",
        "cat": "CatBoost",
        "lr": "Logistic Regression",
        "logistic_regression": "Logistic Regression",
        "lightgbm": "LightGBM",
        "xgboost": "XGBoost",
        "catboost": "CatBoost",
    }

    # Initialize OOF and test probabilities
    oof_probs = {name: np.zeros(len(X)) for name in model_names}
    test_probs_by_model = {name: np.zeros(len(X_test)) for name in model_names}
    metrics_by_model = {name: {"train": [], "val": []} for name in model_names}

    print(f"Starting {n_repeats}x Stratified {k}-Fold CV with model: {model_type}.")

    for i, (train_idx, val_idx) in enumerate(rskf.split(X, y)):
        repeat, fold = divmod(i, k)

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = build_model(model_type, random_state=random_state)
        model.fit(X_train, y_train_fold)

        # Robust extraction: inspect named_estimators_ only if VotingClassifier
        if hasattr(model, "named_estimators_"):
            fitted_models = {"ensemble": model, **dict(model.named_estimators_)}
        else:
            fitted_models = {model_type: model}

        for name, estimator in fitted_models.items():
            if name not in model_names:
                continue

            # Training evaluation
            train_prob = estimator.predict_proba(X_train)[:, 1]
            train_pred = (train_prob >= 0.5).astype(int)
            metrics_by_model[name]["train"].append(
                compute_metrics(y_train_fold, train_pred, train_prob)
            )

            # Validation evaluation
            val_prob = estimator.predict_proba(X_val)[:, 1]
            val_pred = (val_prob >= 0.5).astype(int)
            metrics_by_model[name]["val"].append(
                compute_metrics(y_val_fold, val_pred, val_prob)
            )

            # Accumulate OOF probabilities (normalized across repeats)
            oof_probs[name][val_idx] += val_prob / n_repeats

            # Accumulate individual test probabilities across all folds and repeats
            test_probs_by_model[name] += estimator.predict_proba(X_test)[:, 1] / n_total

        main_key = "ensemble" if model_type == "ensemble" else model_type
        fold_metrics = metrics_by_model[main_key]["val"][-1]
        metrics_text = "".join(f"{n}: {v:.4f} | " for n, v in fold_metrics.items())
        print(f"Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{k} -> {metrics_text}")

    print_val_train_metrics(n_repeats, k, n_total, model_names, metrics_by_model, oof_probs, y, display_names)
    print_oof_metrics(n_repeats, k, n_total, model_names, metrics_by_model, oof_probs, y, display_names)
    print_overfitting_diagnostics(model_names, metrics_by_model, display_names)

    return oof_probs, test_probs_by_model


def print_val_train_metrics(n_repeats: int, k: int, n_total: int, model_names: list, metrics_by_model: dict, oof_probs: dict, y: pd.Series, display_names: dict) -> None:
    print(f"\nAggregated cv results ({n_repeats}x{k} folds = {n_total} evaluations):")
    for dataset in ["val", "train"]:
        print(f"\n{dataset.upper()} METRICS")
        print("----------------------------------------------------------------------")
        for name in model_names:
            fold_metrics = metrics_by_model[name][dataset]
            mean_acc = np.mean([m["accuracy"] for m in fold_metrics])
            std_acc  = np.std([m["accuracy"] for m in fold_metrics])
            mean_f1  = np.mean([m["f1"] for m in fold_metrics])
            mean_auc = np.mean([m["roc_auc"] for m in fold_metrics])
            print(
                f"{display_names.get(name, name):<22} | "
                f"Accuracy: {mean_acc:.4f} (+/- {std_acc:.4f}) | "
                f"F1: {mean_f1:.4f} | "
                f"ROC-AUC: {mean_auc:.4f}"
            )


def print_oof_metrics(n_repeats: int, k: int, n_total: int, model_names: list, metrics_by_model: dict, oof_probs: dict, y: pd.Series, display_names: dict) -> None:
    print("\nOVERALL OOF METRICS (Default 0.50 Threshold)")
    print("----------------------------------------------------------------------")
    for name in model_names:
        oof_pred = (oof_probs[name] >= 0.5).astype(int)
        overall_oof_acc = accuracy_score(y, oof_pred)
        overall_oof_auc = roc_auc_score(y, oof_probs[name])
        print(
            f"{display_names.get(name, name):<22} | "
            f"Accuracy: {overall_oof_acc:.4f} | "
            f"ROC-AUC: {overall_oof_auc:.4f}"
        )

        # The scored validation fold never participates in early stopping.
        stopping_model = build_model(random_state=seed)
        stopping_model.fit(
            fold_X.iloc[fit_idx], y.iloc[fit_idx],
            cat_features=cats,
            eval_set=(fold_X.iloc[stop_idx], y.iloc[stop_idx]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
        )
        iterations = max(1, stopping_model.get_best_iteration() + 1)
        model = build_model(random_state=seed, iterations=iterations)
        model.fit(fold_X.iloc[train_idx], y.iloc[train_idx], cat_features=cats)

def print_overfitting_diagnostics(model_names: list, metrics_by_model: dict, display_names: dict) -> None:
    print("\nOVERFITTING / UNDERFITTING DIAGNOSTICS")
    print("=" * 100)
    for name in model_names:
        train_metrics = metrics_by_model[name]["train"]
        val_metrics = metrics_by_model[name]["val"]

        train_acc = np.mean([m["accuracy"] for m in train_metrics])
        train_f1 = np.mean([m["f1"] for m in train_metrics])
        train_auc = np.mean([m["roc_auc"] for m in train_metrics])

        val_acc = np.mean([m["accuracy"] for m in val_metrics])
        val_f1 = np.mean([m["f1"] for m in val_metrics])
        val_auc = np.mean([m["roc_auc"] for m in val_metrics])

        acc_gap = train_acc - val_acc
        f1_gap = train_f1 - val_f1
        auc_gap = train_auc - val_auc

        if max(acc_gap, f1_gap, auc_gap) >= 0.10:    status = "Possible overfitting"
        elif max(acc_gap, f1_gap, auc_gap) >= 0.05:  status = "Mild overfitting"
        elif min(acc_gap, f1_gap, auc_gap) < -0.05:  status = "Validation exceeds training"
        else:                                        status = "No strong generalization gap"

        print(f"\n{display_names.get(name, name)}")
        print("----------------------------------------------------------------------")
        print(f"Accuracy | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Gap: {acc_gap:+.4f}")
        print(f"F1       | Train: {train_f1:.4f}  | Validation: {val_f1:.4f} | Gap: {f1_gap:+.4f}")
        print(f"ROC-AUC  | Train: {train_auc:.4f} | Validation: {val_auc:.4f} | Gap: {auc_gap:+.4f}")
        print(f"Status: {status}")


# Features excluding ID and target columns
feature_cols = [
    c for c in train_df.columns
    if c not in ["battle_id", "player_won"] and c in ALLOWED_FEATURES
]
missing_allowed_features = [
    feature for feature in ALLOWED_FEATURES if feature not in train_df.columns
]

for col in feature_cols:
    if col not in test_df.columns:
        test_df[col] = 0.0


def export_run_report(
    summary: pd.DataFrame,
    used_features: list[str],
    use_allowed_features: bool,
    output_dir: str | Path = REPORT_DIR,
    selection_history: pd.DataFrame | None = None,
    cv_results: pd.DataFrame | None = None,
    candidate_features: list[str] | None = None,
) -> Path:
    """Save the displayed validation summary and the actual selected predictors."""
    created_at = datetime.now(timezone.utc)
    report_dir = Path(output_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    report_path = report_dir / f"catboost_report_{created_at:%Y%m%dT%H%M%S_%fZ}.md"

# Run Cross-Validation
oof_probabilities, individual_test_probs = train_and_evaluate_cv(
    X=X_train,
    y=y_train,
    X_test=X_test,
    model_type=MODEL_TYPE,
    k=K_FOLDS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

In [ ]:
def optimize_ensemble_and_predict(oof_probs: dict, test_probs_by_model: dict, y_true: pd.Series):
    """
    If multiple models are present (ensemble), optimizes blend weights.
    If a single model is present (e.g., Logistic Regression), tunes the decision threshold directly.
    """
    model_keys = [k for k in ["lgb", "xgb", "cat", "lr", "logistic_regression", "lightgbm", "xgboost", "catboost"] if k in oof_probs]

    # CASE 1: Single Model (Standalone Logistic Regression)
    if len(model_keys) == 1:
        single_key = model_keys[0]
        blended_oof = oof_probs[single_key]
        blended_test = test_probs_by_model[single_key]

        thresholds = np.linspace(0.42, 0.58, 161)
        accs = [accuracy_score(y_true, (blended_oof >= t).astype(int)) for t in thresholds]
        best_threshold = thresholds[np.argmax(accs)]
        best_oof_acc = np.max(accs)
        baseline_acc = accuracy_score(y_true, (blended_oof >= 0.50).astype(int))

        print("=" * 70)
        print(f"SINGLE MODEL THRESHOLD CALIBRATION ({single_key.upper()})")
        print("=" * 70)
        print(f"  Optimal Decision Cutoff     : {best_threshold:.4f}")
        print(f"  OOF Accuracy (Default 0.50) : {baseline_acc:.4f}")
        print(f"  OOF Accuracy (Calibrated)   : {best_oof_acc:.4f} (+{(best_oof_acc - baseline_acc) * 100:.2f}%)")
        print("=" * 70)

        final_test_preds = (blended_test >= best_threshold).astype(int)
        return final_test_preds, blended_test

    # CASE 2: Multi-Model Ensemble
    oof_matrix = np.column_stack([oof_probs[m] for m in model_keys])
    test_matrix = np.column_stack([test_probs_by_model[m] for m in model_keys])

    def brier_loss(weights):
        w = np.array(weights)
        w = np.clip(w, 0, None)
        if np.sum(w) == 0:
            return 1.0
        w = w / np.sum(w)
        blend = oof_matrix @ w
        return np.mean((blend - y_true) ** 2)

    res = minimize(brier_loss, [1.0] * len(model_keys), method="Nelder-Mead")
    best_weights = np.clip(res.x, 0, None)
    best_weights = best_weights / np.sum(best_weights)

    blended_oof = oof_matrix @ best_weights
    blended_test = test_matrix @ best_weights

    thresholds = np.linspace(0.42, 0.58, 161)
    accs = [accuracy_score(y_true, (blended_oof >= t).astype(int)) for t in thresholds]
    best_threshold = thresholds[np.argmax(accs)]
    best_oof_acc = np.max(accs)
    baseline_acc = accuracy_score(y_true, (blended_oof >= 0.50).astype(int))

    print("=" * 70)
    print("DYNAMIC ENSEMBLE OPTIMIZATION & THRESHOLD CALIBRATION")
    print("=" * 70)
    for m, w in zip(model_keys, best_weights):
        print(f"  {m.upper():<4} Weight : {w:.4f} (relative: {w * len(model_keys):.2f})")
    print(f"  Optimal Decision Cutoff     : {best_threshold:.4f}")
    print(f"  OOF Accuracy (Default 0.50) : {baseline_acc:.4f}")
    print(f"  OOF Accuracy (Calibrated)   : {best_oof_acc:.4f} (+{(best_oof_acc - baseline_acc) * 100:.2f}%)")
    print("=" * 70)

    final_test_preds = (blended_test >= best_threshold).astype(int)
    return final_test_preds, blended_test

# Execute calibration
final_test_predictions, final_test_probabilities = optimize_ensemble_and_predict(
    oof_probs=oof_probabilities,
    test_probs_by_model=individual_test_probs,
    y_true=y_train,
)

## 4. Submission File

In [ ]:
def predict_and_save(predictions: np.ndarray, test_battle_ids: pd.Series, output_path: str = SUBMISSION_PATH, sample_path: str = SAMPLE_SUBMISSION_PATH) -> None:
    """
    Format and save the submission file using the required template.
    """
    print(f"Saving predictions to '{output_path}'...")
    sample_submission = pd.read_csv(sample_path)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"ERROR: expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    sample_submission["player_won"] = sample_submission["battle_id"].map(predictions_by_id)

    if sample_submission["player_won"].isna().any():
        raise ValueError(f"ERROR: some battle IDs in '{sample_path}' have no matching prediction.")

    sample_submission["player_won"] = sample_submission["player_won"].astype(int)
    sample_submission.to_csv(output_path, index=False)

    print(f"SUCCESS: file '{output_path}' created.")
    print(sample_submission.head(10))
    print(f"Prediction distribution (0 vs 1): {dict(sample_submission['player_won'].value_counts())}")

# Generate and save final submission using calibrated predictions
predict_and_save(final_test_predictions, test_df["battle_id"], SUBMISSION_PATH, SAMPLE_SUBMISSION_PATH)

## Debug

In [ ]:
display(feature_importance.head(25).to_frame())

if DEBUG:
    print_battle(train_data)
    diagnose_type_chart(type_chart, stats_map, train_data)
    diagnose_fnt_features(train_df)
    diagnose_ko_records(train_data)
    diagnose_species_stats(train_data, test_data)
    diagnose_distinct_fnt(train_data)
    print_features_correlation(X_train, y_train)
